## Overview
This notebook illustrates how to create a vector store from markdown documents, and to use this in a RAG assisted LLM query. Llamaindex has an integration with vllm, and here we use Milvus as the vector store which is also integrated with VLLM. Milvus is performant and has nice features and supports a wide range of retrieval options including variants of hybrid search (text keyword + embeddings).

Important factors in RAG performance to explore in the future:
- Input document data quality (currently not implemented)
- Document chunking strategy (currently not implemented)
- Embedding model choice
- Retrieval strategy, including results re-ranking
- Prompt language - how the LLM is instructed to use the retrieved context and prior knowledge

### Parameters
The pipeline requires markdown document(s), an embedding model, a path to the milvus database, an llm model, a query, a number of documents to retrieve ('top_k'), and a Hugging Face token with approval for the llm and embedding models.

In [11]:
document_path = "./documents/myeloid_raw-depth-2.json"
embedding_name = "BAAI/bge-small-en-v1.5"
db_path = "./milvusdb/milvus_llamaindex.db"
llm_name = "google/medgemma-1.5-4b-it"
query = "What are the diagnostic criteria for myeloproliferative neoplasm NOS?"
top_k = 2 # this parameter might be consequential if the information is spread through multiple document chunks
hf_token = "****************************"

### Installs and imports
No additional libraries, aside from llamaindex and its plugins, are required to run the RAG.

In [12]:
# minimal install
!pip install chromadb llama_index llama-index-embeddings-huggingface llama-index-llms-huggingface
!pip install llama-index-vector-stores-milvus

# install if LLM complains about transformers version
# !pip install --upgrade transformers

I0000 00:00:1788376776.935262    2685 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Defaulting to user installation because normal site-packages is not writeable


I0000 00:00:1788376781.441535    2685 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Defaulting to user installation because normal site-packages is not writeable


In [13]:
from llama_index.core import Document, VectorStoreIndex, Settings, StorageContext
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.response.pprint_utils import pprint_response
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.vector_stores.milvus import MilvusVectorStore

### Load markdown formatted content

In [14]:
# build a dictionary of contents indexed by url - in html and markdown formats
import json
sections = dict()
markdown_docs = []
with open(document_path, "r", encoding="utf-8") as file:
    aggregated = json.load(file)
    for section in aggregated:
        if section["url"] not in sections:
            sections[section["url"]] = {
                "markdown": section["markdown"],
            }
markdown_docs = [Document(text=v["markdown"]) for v in sections.values()] 

### Create milvus vector database

In [15]:
import os
os.environ["HF_TOKEN"] = hf_token
embed_model = HuggingFaceEmbedding(model_name=embedding_name)

vector_store = MilvusVectorStore(
    uri=db_path,
    dim=384,  # Embedding dimension for BAAI/bge-small-en-v1.5
    overwrite=True # when running multiple times, recreate the database from scratch to avoid duplicate entries
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
pipeline = IngestionPipeline(
    transformations=[
        MarkdownNodeParser(),
        embed_model,
    ]
)
nodes = pipeline.run(documents=markdown_docs, show_progress=True, num_workers=None)


# this is a global setting - be cautious and understand that it is persistent until changed
# this line is required - otherwise VectorStoreIndex call tries to access OpenAI model
Settings.embed_model = embed_model

index = VectorStoreIndex(
    nodes, 
    storage_context=storage_context
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

### Query using LLM with RAG

In [16]:
# this is another llamaindex global setting that is required
Settings.llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=512, # increase if the response appears truncated
    model_name=llm_name,
    tokenizer_name=llm_name,
)

query_engine = index.as_query_engine(similarity_top_k=top_k, response_mode="compact")
response = query_engine.query(query)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

I0000 00:00:1788376928.843633    2854 chttp2_transport.cc:1335] ipv4:127.0.0.1:39815: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0000 00:00:1788376928.843668    2854 chttp2_transport.cc:1364] ipv4:127.0.0.1:39815: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


### Show response text and documents sourced

In [9]:
print(response)


The diagnostic criteria for myeloproliferative neoplasm NOS (unclassifiable) are based on the exclusion of other specific myeloproliferative neoplasm types. The criteria are as follows:
1.  **Clinical, laboratory, morphological, and molecular features:** The case must exhibit definite clinical, laboratory, morphological, and molecular features of an MPN.
2.  **Exclusion of specific types:** The features must not meet the diagnostic criteria for any of the specific types of MPN, such as polycythaemia vera, essential thrombocythaemia, or primary myelofibrosis.
3.  **Morphological evidence:** Bone marrow examination must show morphological changes characteristic of an MPN.
4.  **Clonal genetic marker:** The presence of a clonal genetic marker (such as _JAK2_ , _CALR_ , or _MPL_ mutations) is required.
5.  **Diagnosis of exclusion:** MPN-NOS is a diagnosis of exclusion, meaning it is assigned only when other specific MPN types cannot be definitively diagnosed based on the available inform

In [7]:
for node in response.source_nodes:
    print("*" * 80)
    print(f"Node ID: {node.node_id}")
    print(f"Score: {node.score}")
    print(f"Text: {node.text}")
    print(f"Metadata: {node.node.metadata}")

********************************************************************************
Node ID: d432a142-1f80-4e03-ad26-1217e8954443
Score: 0.8628533482551575
Text: ####  Myeloproliferative neoplasm NOS (unclassifiable)![](/assets/Images/add-to-favourites.png)
Definition
Myeloproliferative neoplasm (MPN) not otherwise specified (NOS) is a diagnosis of exclusion for cases that have definite clinical, laboratory, morphological, and molecular features of an MPN but fail to meet the diagnostic criteria for any of the specific types in this disease category.
ICD-O coding
9975/3 Myeloproliferative neoplasm, NOS (unclassifiable)
ICD-11 coding
2A4Z & XH5HH7 Myelodysplastic and myeloproliferative neoplasms, unspecified & Myeloproliferative neoplasm, NOS
Related terminology
_Acceptable:_ myeloproliferative neoplasm, unclassifiable.
Subtype(s)
None
Localization
The blood and bone marrow are the major sites of involvement, but in advanced stages the spleen and liver as the major sites of extramedullary 